# 03 · Preference optimization (DPO)

SFT says "copy this answer". DPO says "**this** answer is better than **that** one".

Data: (prompt, chosen, rejected). Loss: push up log p(chosen) relative to
log p(rejected), measured against a frozen **reference** model so we don't drift:

    loss = -log σ( β · [ (log π(chosen) - log π_ref(chosen)) - (log π(rejected) - log π_ref(rejected)) ] )

New Tinker feature: `forward_backward_custom`. Tinker gives you per-token
logprobs, you write *any* differentiable loss on them in PyTorch.

In [ ]:
import torch.nn.functional as F

from common import *

prefs = datasets.load_dataset("argilla/ultrafeedback-binarized-preferences", split="train").shuffle(seed=0)
BATCH_SIZE, STEPS, LR, BETA, MAX_LEN = 16, 25, 1e-5, 0.1, 512

ex = prefs[0]
show("prompt", ex["instruction"][:500])
show("chosen", ex["chosen_response"][:500])
show("rejected", ex["rejected_response"][:500])

In [ ]:
def response_tokens_and_weights(prompt, response):
    convo = [{"role": "user", "content": prompt}, {"role": "assistant", "content": response}]
    mi, w = renderer.build_supervised_example(convo, train_on_what=renderers.TrainOnWhat.LAST_ASSISTANT_MESSAGE)
    return mi.to_ints()[:MAX_LEN], w.tolist()[:MAX_LEN]

# Frozen reference policy = the starting model.
reference = service_client.create_sampling_client(base_model=MODEL).result()

In [ ]:
# ── LIVE: the DPO loss ────────────────────────────────────────────────────
def dpo_loss(data, logprobs_list):
    """data/logprobs alternate [chosen, rejected, chosen, rejected, ...]."""
    seq_logprob = [(lp * d.loss_fn_inputs["weights"].to_torch()).sum() for d, lp in zip(data, logprobs_list)]
    policy_margin = torch.stack(seq_logprob[0::2]) - torch.stack(seq_logprob[1::2])
    ref_margin = torch.tensor(ref_seq_logprob[0::2]) - torch.tensor(ref_seq_logprob[1::2])
    logits = BETA * (policy_margin - ref_margin)
    loss = -F.logsigmoid(logits).mean()
    return loss, {"accuracy": (logits > 0).float().mean().item(), "margin": logits.mean().item()}

In [ ]:
# ── LIVE: training loop ───────────────────────────────────────────────────
training_client = new_training_client()

for step in range(STEPS):
    rows = prefs.select(range(step * BATCH_SIZE, (step + 1) * BATCH_SIZE))
    seqs = []
    for p, c, r in zip(rows["instruction"], rows["chosen_response"], rows["rejected_response"]):
        seqs += [response_tokens_and_weights(p, c), response_tokens_and_weights(p, r)]
    batch = [make_datum(toks, w) for toks, w in seqs]

    # Reference logprobs: one forward pass on the frozen model, no gradients.
    ref_futures = [reference.compute_logprobs(types.ModelInput.from_ints(toks)) for toks, _ in seqs]
    ref_seq_logprob = [
        sum(lp * wt for lp, wt in zip(f.result()[1:], w[1:])) for f, (_, w) in zip(ref_futures, seqs)
    ]

    result = training_client.forward_backward_custom(batch, dpo_loss).result()
    training_client.optim_step(adam(LR)).result()
    m = result.metrics  # your custom metrics come back here alongside loss:sum
    print(f"step {step:2d}  loss={m['loss:sum']:.3f}  pref_acc={m['accuracy']:.2f}  margin={m['margin']:+.3f}")

**Takeaway:** DPO learns from *comparisons*, which are much easier for humans to
label than ideal answers. But it's still offline: the pairs were written by
other models, not by ours.

---
**Bridge to RL:** what if the model generated its own attempts and we scored them?